# CIMA batch GWAS 后处理 notebook

这个 notebook 用于 **GWAS 跑完之后** 的统一整理。默认流程：

1. 读取 batch GWAS summary
2. 读取 phenotype 原始名 / 安全名映射
3. 汇总每个 phenotype 的核心统计量
4. 计算 lambda GC
5. 筛选值得重点查看的 phenotype
6. 批量绘制 Manhattan / QQ 图
7. 导出 top hits 和 phenotype-level summary

默认假设你前面的 batch GWAS 结果目录结构是：

- `data/results/gwas_batch/per_pheno/<phenotype>/`
- `data/results/gwas_batch/summary/gwas_run_summary.tsv`
- `data/processed/CIMA/phenotype/CIMA_metabolites_lipids_pheno_name_map.tsv`

这版 notebook 尽量保持简单直接，适合你后面继续接 locus / gene 注释。

In [2]:
from pathlib import Path
import math
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ===== 路径设置 =====
PROJECT_ROOT = Path("/data/work/CIMA_multiomics_regulation")

GWAS_BASE = PROJECT_ROOT / "data/results/gwas_all"
PER_PHENO_DIR = GWAS_BASE / "per_pheno"
SUMMARY_DIR = GWAS_BASE / "summary"
PLOT_DIR = GWAS_BASE / "post_gwas_plots"
PLOT_MANHATTAN_DIR = PLOT_DIR / "manhattan"
PLOT_QQ_DIR = PLOT_DIR / "qq"
POST_DIR = GWAS_BASE / "post_gwas"

PHENO_NAME_MAP_FILE = PROJECT_ROOT / "data/processed/CIMA/phenotype/CIMA_metabolites_lipids_pheno_name_map.tsv"
RUN_SUMMARY_FILE = SUMMARY_DIR / "gwas_run_summary.tsv"

for d in [PLOT_DIR, PLOT_MANHATTAN_DIR, PLOT_QQ_DIR, POST_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("GWAS_BASE:", GWAS_BASE)
print("PER_PHENO_DIR:", PER_PHENO_DIR)
print("RUN_SUMMARY_FILE:", RUN_SUMMARY_FILE)
print("PHENO_NAME_MAP_FILE:", PHENO_NAME_MAP_FILE)

PROJECT_ROOT: /data/work/CIMA_multiomics_regulation
GWAS_BASE: /data/work/CIMA_multiomics_regulation/data/results/gwas_all
PER_PHENO_DIR: /data/work/CIMA_multiomics_regulation/data/results/gwas_all/per_pheno
RUN_SUMMARY_FILE: /data/work/CIMA_multiomics_regulation/data/results/gwas_all/summary/gwas_run_summary.tsv
PHENO_NAME_MAP_FILE: /data/work/CIMA_multiomics_regulation/data/processed/CIMA/phenotype/CIMA_metabolites_lipids_pheno_name_map.tsv


## 读取 batch 运行摘要

这里会读取你前面 batch GWAS 产生的 `gwas_run_summary.tsv`，只保留成功的 phenotype。

In [3]:
run_summary = pd.read_csv(RUN_SUMMARY_FILE, sep="\t")
print("run_summary shape:", run_summary.shape)
display(run_summary.head())

if "status" not in run_summary.columns:
    raise ValueError("gwas_run_summary.tsv 缺少 status 列")

done_df = run_summary[run_summary["status"] == "done"].copy()
print("done phenotype count:", done_df.shape[0])

if done_df.empty:
    raise ValueError("没有 status=done 的 phenotype，先检查 batch GWAS 是否真的跑完了")

run_summary shape: (1533, 14)


,phenotype,status,glm_file,add_file,snplist_file,log_file,wrapper_log,n_add_rows,n_snps,start_time,finished_at,elapsed_seconds,error,phenotype_original
0,Cer_d18_1_22_0,done,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,7405576.0,6656848.0,2026-04-17T12:56:31,2026-04-17T12:58:39,128.412035,NaN,Cer d18:1/22:0
1,Cer_d18_1_24_0,done,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,7405576.0,6656848.0,2026-04-17T12:49:32,2026-04-17T12:51:44,131.608883,NaN,Cer d18:1/24:0
2,Cer_d18_1_24_1,done,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,7405576.0,6656848.0,2026-04-17T13:04:25,2026-04-17T13:07:16,170.436648,NaN,Cer d18:1/24:1
3,DAG_16_0_18_3,done,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,7405576.0,6656848.0,2026-04-17T12:58:36,2026-04-17T13:01:33,176.777417,NaN,DAG 16:0-18:3
4,DAG_16_1_18_3,done,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,7405576.0,6656848.0,2026-04-17T13:02:07,2026-04-17T13:04:31,143.552243,NaN,DAG 16:1-18:3


done phenotype count: 516


## 读取 phenotype 名称映射

如果你 batch GWAS 是用安全列名跑的，这一步会把安全名再映射回原始 metabolite/lipid 名，方便最后看结果。

In [4]:
if PHENO_NAME_MAP_FILE.exists():
    name_map = pd.read_csv(PHENO_NAME_MAP_FILE, sep="\t")
    print("name_map shape:", name_map.shape)
    display(name_map.head())
else:
    print("未找到 phenotype name map，后面将只使用 phenotype 列")
    name_map = pd.DataFrame(columns=["old_name", "new_name"])

if not name_map.empty:
    done_df = done_df.merge(
        name_map.rename(columns={"new_name": "phenotype", "old_name": "phenotype_original"}),
        on="phenotype",
        how="left"
    )
else:
    done_df["phenotype_original"] = done_df["phenotype"]

display(done_df.head())

name_map shape: (1551, 2)


,old_name,new_name
0,FID,FID
1,IID,IID
2,O-Phosphoryl-ethanolamine,O_Phosphoryl_ethanolamine
3,Acesulfame,Acesulfame
4,Acetic acid,Acetic_acid


,phenotype,status,glm_file,add_file,snplist_file,log_file,wrapper_log,n_add_rows,n_snps,start_time,finished_at,elapsed_seconds,error,phenotype_original_x,phenotype_original_y
0,Cer_d18_1_22_0,done,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,7405576.0,6656848.0,2026-04-17T12:56:31,2026-04-17T12:58:39,128.412035,NaN,Cer d18:1/22:0,Cer d18:1/22:0
1,Cer_d18_1_24_0,done,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,7405576.0,6656848.0,2026-04-17T12:49:32,2026-04-17T12:51:44,131.608883,NaN,Cer d18:1/24:0,Cer d18:1/24:0
2,Cer_d18_1_24_1,done,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,7405576.0,6656848.0,2026-04-17T13:04:25,2026-04-17T13:07:16,170.436648,NaN,Cer d18:1/24:1,Cer d18:1/24:1
3,DAG_16_0_18_3,done,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,7405576.0,6656848.0,2026-04-17T12:58:36,2026-04-17T13:01:33,176.777417,NaN,DAG 16:0-18:3,DAG 16:0-18:3
4,DAG_16_1_18_3,done,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,/data/work/CIMA_multiomics_regulation/data/res...,7405576.0,6656848.0,2026-04-17T13:02:07,2026-04-17T13:04:31,143.552243,NaN,DAG 16:1-18:3,DAG 16:1-18:3


## 定义后处理函数

这里包括：

- 读单个 `.add` 文件
- 计算 lambda GC
- 提取 top hit
- 统计不同 P 阈值下的 hit 数量
- 画 Manhattan 图
- 画 QQ 图

In [5]:
def check_exists(fp: Path):
    if not fp.exists():
        raise FileNotFoundError(f"文件不存在: {fp}")

def find_add_file(pheno_dir: Path, phenotype: str):
    candidates = sorted(pheno_dir.glob(f"{phenotype}*.add"))
    if len(candidates) == 0:
        raise FileNotFoundError(f"未找到 add 文件: {pheno_dir}")
    return candidates[0]

def read_add_file(add_file: Path):
    df = pd.read_csv(add_file, sep="\t", low_memory=False)

    required = ["#CHROM", "POS", "ID", "P"]
    for c in required:
        if c not in df.columns:
            raise ValueError(f"{add_file} 缺少列: {c}")

    df = df.copy()
    df["#CHROM"] = df["#CHROM"].astype(str).str.replace("^chr", "", regex=True)
    df["POS"] = pd.to_numeric(df["POS"], errors="coerce")
    df["P"] = pd.to_numeric(df["P"], errors="coerce")

    if "BETA" in df.columns:
        df["BETA"] = pd.to_numeric(df["BETA"], errors="coerce")
    if "SE" in df.columns:
        df["SE"] = pd.to_numeric(df["SE"], errors="coerce")

    valid_chr = [str(i) for i in range(1, 23)] + ["X", "Y", "MT", "M"]
    df = df[df["#CHROM"].isin(valid_chr)].copy()
    df = df[df["POS"].notna() & df["P"].notna()].copy()
    df = df[(df["P"] > 0) & (df["P"] <= 1)].copy()

    return df

def calc_lambda_gc_from_p(pvals):
    pvals = np.asarray(pvals, dtype=float)
    pvals = pvals[np.isfinite(pvals)]
    pvals = pvals[(pvals > 0) & (pvals < 1)]

    if pvals.size == 0:
        return np.nan

    # chi-square(1) quantile from p-value without scipy:
    # chi2_1 quantile = (norm.isf(p/2))^2
    # use numpy approximation through erfcinv is unavailable, so fallback to pandas if scipy absent is hard.
    # here use scipy if available, else return nan.
    try:
        from scipy.stats import chi2
        chisq = chi2.isf(pvals, df=1)
        lambda_gc = np.median(chisq) / 0.4549364
        return float(lambda_gc)
    except Exception:
        return np.nan

def make_manhattan_df(df):
    chr_order = [str(i) for i in range(1, 23)] + ["X", "Y", "MT", "M"]
    chr_to_num = {c: i+1 for i, c in enumerate(chr_order)}

    plot_df = df.copy()
    plot_df["chr_num"] = plot_df["#CHROM"].map(chr_to_num)
    plot_df = plot_df[plot_df["chr_num"].notna()].copy()
    plot_df["chr_num"] = plot_df["chr_num"].astype(int)
    plot_df = plot_df.sort_values(["chr_num", "POS"]).reset_index(drop=True)
    plot_df["minus_log10_p"] = -np.log10(plot_df["P"])

    offsets = {}
    current = 0
    centers = []

    for chrom in chr_order:
        sub = plot_df[plot_df["#CHROM"] == chrom]
        if sub.empty:
            continue
        min_pos = sub["POS"].min()
        max_pos = sub["POS"].max()
        offsets[chrom] = current - min_pos
        center = current + (max_pos - min_pos) / 2
        centers.append((chrom, center))
        current += (max_pos - min_pos) + 1e6

    plot_df["plot_pos"] = plot_df.apply(lambda x: x["POS"] + offsets[x["#CHROM"]], axis=1)
    center_df = pd.DataFrame(centers, columns=["chrom", "center"])
    return plot_df, center_df

def plot_manhattan(df, phenotype_label, out_file):
    plot_df, center_df = make_manhattan_df(df)
    if plot_df.empty:
        return

    plt.figure(figsize=(14, 5))
    for chrom_i, (chrom, sub) in enumerate(plot_df.groupby("#CHROM", sort=False)):
        plt.scatter(
            sub["plot_pos"],
            sub["minus_log10_p"],
            s=6,
            alpha=0.7
        )

    plt.axhline(-np.log10(5e-8), linestyle="--", linewidth=1)
    plt.axhline(-np.log10(1e-5), linestyle="--", linewidth=1)

    if not center_df.empty:
        plt.xticks(center_df["center"], center_df["chrom"], rotation=0)

    plt.xlabel("Chromosome")
    plt.ylabel("-log10(P)")
    plt.title(f"Manhattan: {phenotype_label}")
    plt.tight_layout()
    plt.savefig(out_file, dpi=200, bbox_inches="tight")
    plt.close()

def plot_qq(df, phenotype_label, out_file):
    pvals = df["P"].dropna().values
    pvals = pvals[(pvals > 0) & (pvals <= 1)]

    if pvals.size == 0:
        return

    obs = -np.log10(np.sort(pvals))
    exp = -np.log10((np.arange(1, len(obs) + 1) - 0.5) / len(obs))

    maxv = max(obs.max(), exp.max())

    plt.figure(figsize=(5, 5))
    plt.scatter(exp, obs, s=8, alpha=0.7)
    plt.plot([0, maxv], [0, maxv], linewidth=1)
    plt.xlabel("Expected -log10(P)")
    plt.ylabel("Observed -log10(P)")
    plt.title(f"QQ: {phenotype_label}")
    plt.tight_layout()
    plt.savefig(out_file, dpi=200, bbox_inches="tight")
    plt.close()

def summarize_one_pheno(phenotype, phenotype_original=None):
    pheno_dir = PER_PHENO_DIR / phenotype
    add_file = find_add_file(pheno_dir, phenotype)
    df = read_add_file(add_file)

    if df.empty:
        return {
            "phenotype": phenotype,
            "phenotype_original": phenotype_original,
            "add_file": str(add_file),
            "n_snps": 0,
            "min_p": np.nan,
            "lambda_gc": np.nan,
            "n_p_lt_1e5": 0,
            "n_p_lt_1e6": 0,
            "n_p_lt_5e8": 0,
            "top_chr": np.nan,
            "top_pos": np.nan,
            "top_id": np.nan,
            "top_beta": np.nan,
            "top_se": np.nan,
        }, pd.DataFrame()

    top = df.sort_values("P", ascending=True).iloc[0]

    row = {
        "phenotype": phenotype,
        "phenotype_original": phenotype_original,
        "add_file": str(add_file),
        "n_snps": int(df.shape[0]),
        "min_p": float(df["P"].min()),
        "lambda_gc": calc_lambda_gc_from_p(df["P"].values),
        "n_p_lt_1e5": int((df["P"] < 1e-5).sum()),
        "n_p_lt_1e6": int((df["P"] < 1e-6).sum()),
        "n_p_lt_5e8": int((df["P"] < 5e-8).sum()),
        "top_chr": top["#CHROM"],
        "top_pos": int(top["POS"]) if pd.notna(top["POS"]) else np.nan,
        "top_id": top["ID"],
        "top_beta": float(top["BETA"]) if ("BETA" in df.columns and pd.notna(top.get("BETA", np.nan))) else np.nan,
        "top_se": float(top["SE"]) if ("SE" in df.columns and pd.notna(top.get("SE", np.nan))) else np.nan,
    }

    top_hits = df.sort_values("P", ascending=True).head(10).copy()
    top_hits["phenotype"] = phenotype
    top_hits["phenotype_original"] = phenotype_original

    return row, top_hits

## 先做 phenotype-level summary

这里会遍历所有 `status=done` 的 phenotype，生成一个总 summary 表，并导出每个 phenotype 的 top 10 hits。

In [6]:
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
import os
import pandas as pd
from tqdm.auto import tqdm

N_WORKERS = min(64, os.cpu_count() or 1)

if "phenotype_original" in done_df.columns:
    records = done_df[["phenotype", "phenotype_original"]].copy()
    records["phenotype_original"] = records["phenotype_original"].fillna(records["phenotype"])
else:
    records = done_df[["phenotype"]].copy()
    records["phenotype_original"] = records["phenotype"]

task_records = records.to_dict("records")


def _worker(task):
    phenotype = task["phenotype"]
    phenotype_original = task["phenotype_original"]

    try:
        row, top_hits = summarize_one_pheno(phenotype, phenotype_original)

        if row is None:
            row = {"phenotype": phenotype, "phenotype_original": phenotype_original}
        if top_hits is None:
            top_hits = pd.DataFrame()

        return {
            "ok": True,
            "row": row,
            "top_hits": top_hits,
            "error": None,
        }

    except Exception as e:
        return {
            "ok": False,
            "row": None,
            "top_hits": None,
            "error": {
                "phenotype": phenotype,
                "phenotype_original": phenotype_original,
                "error": str(e),
            },
        }


summary_rows = []
top_hits_list = []
errors = []

with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
    futures = [ex.submit(_worker, task) for task in task_records]

    for fut in tqdm(as_completed(futures), total=len(futures), desc="post-gwas summary"):
        res = fut.result()

        if res["ok"]:
            summary_rows.append(res["row"])
            if isinstance(res["top_hits"], pd.DataFrame) and (not res["top_hits"].empty):
                top_hits_list.append(res["top_hits"])
        else:
            errors.append(res["error"])

summary2 = pd.DataFrame(summary_rows)
if not summary2.empty:
    summary2 = summary2.sort_values(["min_p", "phenotype"], ascending=[True, True]).reset_index(drop=True)

if top_hits_list:
    top_hits_df = pd.concat(top_hits_list, axis=0, ignore_index=True)
    top_hits_df = top_hits_df.sort_values(["P", "phenotype"], ascending=[True, True]).reset_index(drop=True)
else:
    top_hits_df = pd.DataFrame()

error_df = pd.DataFrame(errors)

summary2_file = POST_DIR / "phenotype_level_summary.tsv"
top_hits_file = POST_DIR / "all_phenotype_top10_hits.tsv"
error_file = POST_DIR / "post_gwas_errors.tsv"

summary2.to_csv(summary2_file, sep="\t", index=False)
top_hits_df.to_csv(top_hits_file, sep="\t", index=False)
error_df.to_csv(error_file, sep="\t", index=False)

print("summary2 saved:", summary2_file)
print("top hits saved:", top_hits_file)
print("errors saved:", error_file)
print("summary2 shape:", summary2.shape)
print("top_hits_df shape:", top_hits_df.shape)
print("error_df shape:", error_df.shape)

display(summary2.head(20))

post-gwas summary:   0%|          | 0/516 [00:00<?, ?it/s]

## 看整体结果分布

这里先看一下：

- 最小 P 最强的 phenotype
- `lambda_gc` 分布
- 达到 suggestive / genome-wide significance 的 phenotype 有多少

In [ ]:
print("min_p 最小的前20个 phenotype:")
display(
    summary2[[
        "phenotype", "phenotype_original", "n_snps", "min_p", "lambda_gc",
        "n_p_lt_1e5", "n_p_lt_1e6", "n_p_lt_5e8",
        "top_chr", "top_pos", "top_id"
    ]].head(20)
)

print("\n有 suggestive signals (min_p < 1e-5) 的 phenotype 数量:",
      int((summary2["min_p"] < 1e-5).sum()))
print("有 genome-wide significant hits (min_p < 5e-8) 的 phenotype 数量:",
      int((summary2["min_p"] < 5e-8).sum()))

print("\nlambda_gc 描述统计:")
display(summary2["lambda_gc"].describe())

In [ ]:
plt.figure(figsize=(6, 4))
summary2["lambda_gc"].dropna().hist(bins=40)
plt.xlabel("lambda_gc")
plt.ylabel("Count")
plt.title("Distribution of lambda_gc across phenotypes")
plt.tight_layout()
plt.show()

## 筛选值得作图的 phenotype

这里给一个实用默认规则：

- `min_p < 1e-5`
- `lambda_gc` 在 0.9 到 1.1 之间（避免明显膨胀或过保守）
- `n_snps > 0`

你可以自行改阈值。

In [ ]:
PLOT_MIN_P = 1e-5
LAMBDA_LOW = 0.90
LAMBDA_HIGH = 1.10

plot_candidates = summary2[
    (summary2["n_snps"] > 0) &
    (summary2["min_p"] < PLOT_MIN_P) &
    (
        summary2["lambda_gc"].isna() |
        ((summary2["lambda_gc"] >= LAMBDA_LOW) & (summary2["lambda_gc"] <= LAMBDA_HIGH))
    )
].copy()

plot_candidates = plot_candidates.sort_values(["min_p", "phenotype"]).reset_index(drop=True)

plot_candidate_file = POST_DIR / "plot_candidates.tsv"
plot_candidates.to_csv(plot_candidate_file, sep="\t", index=False)

print("plot candidate count:", plot_candidates.shape[0])
print("saved:", plot_candidate_file)
display(plot_candidates.head(30))

## 批量画 Manhattan / QQ 图

默认只对 `plot_candidates` 作图。

如果你想更严格，只看 top 20，可以在下面先 `head(20)`。

In [ ]:
TO_PLOT = plot_candidates.copy()

print("准备作图 phenotype 数量:", TO_PLOT.shape[0])

plot_errors = []

for i, (_, r) in enumerate(TO_PLOT.iterrows(), start=1):
    phenotype = r["phenotype"]
    phenotype_original = r.get("phenotype_original", phenotype)
    pheno_dir = PER_PHENO_DIR / phenotype

    try:
        add_file = find_add_file(pheno_dir, phenotype)
        df = read_add_file(add_file)

        label = phenotype_original if pd.notna(phenotype_original) else phenotype
        safe_label = re.sub(r"[^A-Za-z0-9._-]+", "_", str(phenotype))

        manhattan_file = PLOT_MANHATTAN_DIR / f"{safe_label}.manhattan.png"
        qq_file = PLOT_QQ_DIR / f"{safe_label}.qq.png"

        plot_manhattan(df, label, manhattan_file)
        plot_qq(df, label, qq_file)

        if i <= 10 or i % 20 == 0:
            print(f"[{i}/{TO_PLOT.shape[0]}] done: {phenotype}")
    except Exception as e:
        plot_errors.append({
            "phenotype": phenotype,
            "phenotype_original": phenotype_original,
            "error": str(e)
        })

plot_error_df = pd.DataFrame(plot_errors)
plot_error_file = POST_DIR / "plot_errors.tsv"
plot_error_df.to_csv(plot_error_file, sep="\t", index=False)

print("plot errors:", plot_error_df.shape[0])
print("plot error file:", plot_error_file)

## 看 top phenotype 的图

这里展示最前面的几个 phenotype。  
如果 `plot_candidates` 很少，这里就正好都看了；如果很多，你可以换成自己关心的 phenotype。

In [ ]:
from IPython.display import Image, display

top_show = plot_candidates.head(5).copy()

for _, r in top_show.iterrows():
    phenotype = r["phenotype"]
    phenotype_original = r.get("phenotype_original", phenotype)
    safe_label = re.sub(r"[^A-Za-z0-9._-]+", "_", str(phenotype))

    manhattan_file = PLOT_MANHATTAN_DIR / f"{safe_label}.manhattan.png"
    qq_file = PLOT_QQ_DIR / f"{safe_label}.qq.png"

    print("=" * 80)
    print("phenotype:", phenotype_original)
    print("min_p:", r["min_p"], "lambda_gc:", r["lambda_gc"])

    if manhattan_file.exists():
        display(Image(filename=str(manhattan_file)))
    if qq_file.exists():
        display(Image(filename=str(qq_file)))

## 可选：导出更严格的候选 phenotype

这里给三个层级：

- `significant`: `min_p < 5e-8`
- `suggestive`: `5e-8 <= min_p < 1e-5`
- `null_or_weak`: `min_p >= 1e-5`

In [ ]:
summary3 = summary2.copy()

summary3["signal_level"] = np.where(
    summary3["min_p"] < 5e-8,
    "significant",
    np.where(summary3["min_p"] < 1e-5, "suggestive", "null_or_weak")
)

signal_file = POST_DIR / "phenotype_signal_levels.tsv"
summary3.to_csv(signal_file, sep="\t", index=False)

print("saved:", signal_file)
display(summary3["signal_level"].value_counts(dropna=False))
display(summary3.head(20))

## 这一步跑完后你下一步一般做什么

推荐顺序：

1. 先看 `phenotype_level_summary.tsv`
2. 重点看 `min_p` 最小、`lambda_gc` 正常的 phenotype
3. 再看对应 Manhattan / QQ 图
4. 对有信号的 phenotype 提取独立 lead SNP / locus
5. 做 gene 注释、和 eQTL / pQTL / caQTL 结果对照

也就是说，**不是 1533 个 phenotype 全都盯图看**，而是先 summary，再重点作图，再做 locus-level 整理。